# 🧹 2. Notebook: Data Cleaning Pipeline
## ERP Sales Analytics - Shoebadoo E-Commerce Data

### Umfang

**Eingabe:** Rohe Datensätze aus „01_data_exploration.ipynb“  
**Ausgabe:** Bereinigte, angereicherte Datensätze, bereit für die Qualitätsvalidierung

**Wichtige Aktivitäten:**
- Duplikate entfernen und Datenintegrität validieren
- Fehlende Werte systematisch behandeln (NULL-Normalisierung)
- Marke, Kategorie und Preis aus Produktbeschreibungen mithilfe von NLP extrahieren
- Abgeleitete Felder berechnen (total_amount = Preis × Menge)
- Bereinigte Datensätze für die nachgelagerte Analyse speichern

### Identifizierte Datenprobleme

| Table | Field | Missing | Impact |
|-------|-------|---------|--------|
| PRODUCTS | product_name | 19.2% | Product identification incomplete |
| PRODUCTS | category | 29.8% | Category segmentation limited |
| PRODUCTS | brand | 33.2% | Brand analysis incomplete |
| PRODUCTS | price | 9.2% | Revenue calculation blocked |
| SALES | total_amount | 9.1% | 39,934 rows need calculation |
| RETURNS | refunded_amount | 9.2% | Refund analysis incomplete |



## 1. Setup & Imports

In [1]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, isnan, count, sum as spark_sum, 
    regexp_extract, upper, trim, coalesce, lit,
    length, lower, regexp_replace
)
from pyspark.sql.types import *
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports erfolgreich!")

✅ Imports erfolgreich!


## 2. Spark Session erstellen
Erstellt eine Spark-Session und unterdrückt warnings, die hier eigentlich unnötig wären.

In [2]:
# Warnings zu unterdrücken:
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("py4j").setLevel(logging.ERROR)


# Spark Session erstellen
spark = SparkSession.builder \
    .appName("ERP-Sales-Data-Cleaning") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print(f"✅ Spark Session erstellt!")
print(f"   Spark Version: {spark.version}")
print(f"   App Name: {spark.sparkContext.appName}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/07 09:39:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark Session erstellt!
   Spark Version: 3.5.0
   App Name: ERP-Sales-Data-Cleaning


## 3. Daten laden 

Wir laden die Daten, mit denen wir im ersten Notebook gearbeitet haben.

In [3]:
# Pfad-Konfiguration 
INPUT_PATH = "/app/data/cleaned/1_data_exploration"   # ← Output von Notebook 1
OUTPUT_PATH = "/app/data/cleaned/2_data_cleaning"     # ← Finale bereinigte Daten


print(f"📂 Lade Daten aus: {INPUT_PATH}")
print("-" * 80)

try:
    # Spark liest Parquet-Ordner (nicht Dateien!)
    customers_df = spark.read.parquet(f"{INPUT_PATH}/customers_clean.parquet")
    print(f"✅ customers_df: {customers_df.count():,} rows")
    
    products_df = spark.read.parquet(f"{INPUT_PATH}/products_clean.parquet")
    print(f"✅ products_df:  {products_df.count():,} rows")
    
    sales_df = spark.read.parquet(f"{INPUT_PATH}/sales_clean.parquet")
    print(f"✅ sales_df:     {sales_df.count():,} rows")
    
    returns_df = spark.read.parquet(f"{INPUT_PATH}/returns_clean.parquet")
    print(f"✅ returns_df:   {returns_df.count():,} rows")
    
    print("\n🎉 Alle Dateien erfolgreich geladen!")
    
except Exception as e:
    print(f"❌ Fehler: {e}")
    print("\n💡 Prüfe ob die Dateien existieren:")
    print(f"   ls -la {INPUT_PATH}")

📂 Lade Daten aus: /app/data/cleaned/1_data_exploration
--------------------------------------------------------------------------------
✅ customers_df: 8,000 rows
✅ products_df:  500 rows
✅ sales_df:     437,896 rows
✅ returns_df:   43,789 rows

🎉 Alle Dateien erfolgreich geladen!


## 4. Duplikate entfernen (Verification)

Obwohl wir aus der Explorativen Datenanalyse Report wissen, dass **keine Duplikate** vorhanden sind, führen wir dies als Best Practice durch.

In [4]:
print("🔍 Entferne Duplikate basierend auf Primary Keys...\n")

# Vor der Deduplizierung
customers_before = customers_df.count()
products_before = products_df.count()
sales_before = sales_df.count()
returns_before = returns_df.count()

# Duplikate entfernen basierend auf Primary Keys
customers_clean = customers_df.dropDuplicates(["customer_id"])
products_clean = products_df.dropDuplicates(["product_id"])
sales_clean = sales_df.dropDuplicates(["sale_id"])
returns_clean = returns_df.dropDuplicates(["return_id"])

# Nach der Deduplizierung
customers_after = customers_clean.count()
products_after = products_clean.count()
sales_after = sales_clean.count()
returns_after = returns_clean.count()

# Report
print(f"CUSTOMERS: {customers_before:,} → {customers_after:,} ({customers_before - customers_after} removed)")
print(f"PRODUCTS:  {products_before:,} → {products_after:,} ({products_before - products_after} removed)")
print(f"SALES:     {sales_before:,} → {sales_after:,} ({sales_before - sales_after} removed)")
print(f"RETURNS:   {returns_before:,} → {returns_after:,} ({returns_before - returns_after} removed)")

print("\n✅ Deduplizierung abgeschlossen!")

🔍 Entferne Duplikate basierend auf Primary Keys...

CUSTOMERS: 8,000 → 8,000 (0 removed)
PRODUCTS:  500 → 500 (0 removed)
SALES:     437,896 → 437,896 (0 removed)
RETURNS:   43,789 → 43,789 (0 removed)

✅ Deduplizierung abgeschlossen!


## 5. Missing Values - Initial Fills
Wir gehen hier Schritt für Schritt durch. Es gibt nämlich **verschiedene Arten von fehlenden Werten** in den Parquets.

Diese sind:
- NULL-Werte oder auch NONE in Python
- NaN-Werte oder auch Not a Number - nur bei Float/Double
- Zero also 0 oder 0.0

Wir füllen also erstmal **Text-Felder/Strings mit "UNKNOWN"** um diese zu flaggen und für spätere NLP Extraction.
Danach schauen wir, dass wir alle **numerischen Felder, die fehlen auf NULL / NONE** konvertieren und normalisieren, um damit später besser arbeiten zu können

### 5.1. Text-Felder mit 'UNKNOWN' füllen
- Füllt alle String-Felder mit 'UNKNOWN'
- Zeigt vorher/nachher Statistiken
- Lässt numerische Felder bewusst aus

In [5]:
print("🔧 Fülle fehlende Werte mit Platzhaltern...\n")

# CUSTOMERS (eigentlich keine NULLs vorhanden, aber als Best Practice defensive Programmierung)
customers_clean = customers_clean.fillna({
    'first_name': 'UNKNOWN',
    'last_name': 'UNKNOWN',
    'email': 'unknown@example.com',
    'country': 'UNKNOWN'
})
print("✅ CUSTOMERS: Platzhalter gesetzt")

# PRODUCTS (die Tabelle mit den meisten NULL-Werten!)
# Produktname 19.2%, category 29.8%, brand 33.2%
products_clean = products_clean.fillna({
    'product_name': 'UNKNOWN',
    'category': 'UNKNOWN',
    'brand': 'UNKNOWN',
    # price bleibt NULL - wird später berechnet/gefüllt!
})
print("✅ PRODUCTS: Platzhalter gesetzt (werden später durch NLP ersetzt)")

# SALES
sales_clean = sales_clean.fillna({
    'channel': 'UNKNOWN',
    'payment_method': 'UNKNOWN'
    # total_amount bleibt NULL/NaN - wird in Sektion 6 berechnet!
})
print("✅ SALES: Platzhalter gesetzt")

# RETURNS
returns_clean = returns_clean.fillna({
    'return_reason': 'UNKNOWN'
    # refunded_amount bleibt NULL - business logic entscheidet später
})
print("✅ RETURNS: Platzhalter gesetzt")

print("\n🎉 Initiale Platzhalter gesetzt!")

🔧 Fülle fehlende Werte mit Platzhaltern...

✅ CUSTOMERS: Platzhalter gesetzt
✅ PRODUCTS: Platzhalter gesetzt (werden später durch NLP ersetzt)
✅ SALES: Platzhalter gesetzt
✅ RETURNS: Platzhalter gesetzt

🎉 Initiale Platzhalter gesetzt!


### 5.2 Missing Values - überprüfen obs geklappt hat
Wir wissen vom vorherigen Report, dass:


1. ⚠️ PRODUCTS.product_name: 19.2% NULL
2. ⚠️ PRODUCTS.category: 29.8% NULL
3. ⚠️ PRODUCTS.price: 9.2% NULL
4. ⚠️ PRODUCTS.brand: 33.2% NULL
5. ⚠️ SALES.total_amount: 9.1% NULL
6. ⚠️ RETURNS.refunded_amount: 9.2% NULL


Deswegen überprüfen wir mal products, sales und returns

In [6]:
from pyspark.sql.functions import col

# Liste aller Spalten
columns = products_clean.columns

# Bedingung: irgendeine Spalte enthält 'UNKNOWN'
condition = None
for c in columns:
    cond_col = col(c).cast("string").contains("UNKNOWN")
    condition = cond_col if condition is None else (condition | cond_col)

# Filter anwenden und 5 Zeilen anzeigen
products_clean.filter(condition).show(5, truncate=False)


+----------+-------------------+--------+------+-------+---------------------------------------------------------------------------------------------------+
|product_id|product_name       |category|price |brand  |description                                                                                        |
+----------+-------------------+--------+------+-------+---------------------------------------------------------------------------------------------------+
|502       |UNKNOWN            |UNKNOWN |NaN   |UNKNOWN|Das beliebte Produkt der Marke Puma aus der Kategorie Sport jetzt für nur XX,XX Euro erhältlich!   |
|505       |UNKNOWN            |UNKNOWN |14.76 |UNKNOWN|Das beliebte Produkt der Marke Bench aus der Kategorie Zubehör jetzt für nur 14.76 Euro erhältlich!|
|509       |Boden Alte Spitze  |Sport   |273.82|UNKNOWN|Sein Vater Fenster einigen groß bald die Milch Fußball.                                            |
|512       |UNKNOWN            |UNKNOWN |163.6 |UNKNOWN|Da

### 5.3 Numerische Felder: NaN → NULL normalisieren 
- Konvertiert alle NaN zu NULL
- Konvertiert auch Zero (bei total_amount) zu NULL
- Einheitliche Behandlung aller fehlenden numerischen Werte

In [7]:
print("REPORT: NaN zu NULL konvertieren")
print("-" * 80)
print("⚠️  NaN-Werte sind problematisch und inkonsistent")
print("✅ Wir konvertieren alle NaN → NULL für einheitliche Behandlung")
print()

# ============================================================================
# PRODUCTS: price normalisieren
# ============================================================================
print("🔹 PRODUCTS: Normalisiere 'price'")

# Zähle verschiedene "fehlende" Typen VOR Normalisierung
price_null_before = products_clean.filter(col("price").isNull()).count()
price_nan_before = products_clean.filter(isnan(col("price"))).count()

print(f"   Vor Normalisierung:")
print(f"      NULL: {price_null_before:,}")
print(f"      NaN:  {price_nan_before:,}")

# Normalisiere: NaN → NULL
products_clean = products_clean.withColumn(
    "price",
    when(isnan(col("price")), lit(None))  # NaN wird zu NULL
    .otherwise(col("price"))  # Andere Werte bleiben
)

# Zähle NACH Normalisierung
price_null_after = products_clean.filter(col("price").isNull()).count()
price_nan_after = products_clean.filter(isnan(col("price"))).count()

print(f"   Nach Normalisierung:")
print(f"      NULL: {price_null_after:,}")
print(f"      NaN:  {price_nan_after:,}")
print(f"   ✅ {price_nan_before:,} NaN → NULL konvertiert")
print()

# ============================================================================
# SALES: total_amount normalisieren
# ============================================================================
print("🔹 SALES: Normalisiere 'total_amount'")

# Zähle verschiedene "fehlende" Typen VOR Normalisierung
total_null_before = sales_clean.filter(col("total_amount").isNull()).count()
total_nan_before = sales_clean.filter(isnan(col("total_amount"))).count()
total_zero_before = sales_clean.filter(col("total_amount") == 0).count()

print(f"   Vor Normalisierung:")
print(f"      NULL: {total_null_before:,}")
print(f"      NaN:  {total_nan_before:,}")
print(f"      Zero: {total_zero_before:,}")

# Normalisiere: NaN → NULL, Zero → NULL (da Zero = "fehlend" hier)
sales_clean = sales_clean.withColumn(
    "total_amount",
    when(isnan(col("total_amount")), lit(None))  # NaN → NULL
    .when(col("total_amount") == 0, lit(None))    # 0 → NULL (da fehlend)
    .otherwise(col("total_amount"))
)

# Zähle NACH Normalisierung
total_null_after = sales_clean.filter(col("total_amount").isNull()).count()
total_nan_after = sales_clean.filter(isnan(col("total_amount"))).count()

print(f"   Nach Normalisierung:")
print(f"      NULL: {total_null_after:,}")
print(f"      NaN:  {total_nan_after:,}")
print(f"   ✅ Alle NaN und Zero → NULL konvertiert")
print()

# ============================================================================
# RETURNS: refunded_amount normalisieren
# ============================================================================
print("🔹 RETURNS: Normalisiere 'refunded_amount'")

# Zähle VOR
refund_null_before = returns_clean.filter(col("refunded_amount").isNull()).count()
refund_nan_before = returns_clean.filter(isnan(col("refunded_amount"))).count()

print(f"   Vor Normalisierung:")
print(f"      NULL: {refund_null_before:,}")
print(f"      NaN:  {refund_nan_before:,}")

# Normalisiere
returns_clean = returns_clean.withColumn(
    "refunded_amount",
    when(isnan(col("refunded_amount")), lit(None))
    .otherwise(col("refunded_amount"))
)

# Zähle NACH
refund_null_after = returns_clean.filter(col("refunded_amount").isNull()).count()

print(f"   Nach Normalisierung:")
print(f"      NULL: {refund_null_after:,}")
print(f"   ✅ Normalisierung abgeschlossen")
print()

print("=" * 80)
print("✅ Numerische Felder normalisiert (alle NaN → NULL)!")
print("=" * 80)
print()

REPORT: NaN zu NULL konvertieren
--------------------------------------------------------------------------------
⚠️  NaN-Werte sind problematisch und inkonsistent
✅ Wir konvertieren alle NaN → NULL für einheitliche Behandlung

🔹 PRODUCTS: Normalisiere 'price'
   Vor Normalisierung:
      NULL: 0
      NaN:  46
   Nach Normalisierung:
      NULL: 46
      NaN:  0
   ✅ 46 NaN → NULL konvertiert

🔹 SALES: Normalisiere 'total_amount'
   Vor Normalisierung:
      NULL: 0
      NaN:  39,934
      Zero: 0
   Nach Normalisierung:
      NULL: 39,934
      NaN:  0
   ✅ Alle NaN und Zero → NULL konvertiert

🔹 RETURNS: Normalisiere 'refunded_amount'
   Vor Normalisierung:
      NULL: 0
      NaN:  4,008
   Nach Normalisierung:
      NULL: 4,008
   ✅ Normalisierung abgeschlossen

✅ Numerische Felder normalisiert (alle NaN → NULL)!



### 5.4 Final Verification
- Zeigt alle verbleibenden NULL-Werte
- Klare Übersicht pro Tabelle
- Dokumentiert was noch zu tun ist

In [8]:
print("🔍 Schritt 3: Final Verification - Übersicht fehlender Werte")
print("=" * 80)

def count_missing_values(df, table_name):
    """Zählt NULL-Werte für jede Spalte"""
    print(f"\n📊 {table_name}:")
    print("-" * 80)
    
    has_nulls = False
    for col_name in df.columns:
        null_count = df.filter(col(col_name).isNull()).count()
        if null_count > 0:
            percentage = (null_count / df.count()) * 100
            print(f"   {col_name:20} {null_count:>8,} NULL ({percentage:>5.1f}%)")
            has_nulls = True
    
    if not has_nulls:
        print("   ✅ Keine NULL-Werte!")
    
    return has_nulls

# Prüfe alle Tabellen
count_missing_values(customers_clean, "CUSTOMERS")
count_missing_values(products_clean, "PRODUCTS")
count_missing_values(sales_clean, "SALES")
count_missing_values(returns_clean, "RETURNS")

print("\n" + "=" * 80)
print("✅ SEKTION 5 ABGESCHLOSSEN: Initial Fill & Normalisierung")
print("=" * 80)
print()
print("📝 Zusammenfassung:")
print("   ✅ Text-Felder mit 'UNKNOWN' gefüllt (für NLP-Extraktion)")
print("   ✅ Numerische Felder auf NULL normalisiert (NaN → NULL)")
print("   ✅ Verbleibende NULLs sind GEWOLLT für spätere Berechnungen")
print()
print("➡️  Nächste Schritte:")
print("   📌 Sektion 6: NLP-Extraktion für 'UNKNOWN' Werte")
print("   📌 Sektion 7: NLP-Insertion für 'UNKNOWN' Werte")
print()

🔍 Schritt 3: Final Verification - Übersicht fehlender Werte

📊 CUSTOMERS:
--------------------------------------------------------------------------------
   ✅ Keine NULL-Werte!

📊 PRODUCTS:
--------------------------------------------------------------------------------
   price                      46 NULL (  9.2%)

📊 SALES:
--------------------------------------------------------------------------------


25/11/07 09:39:25 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


   total_amount           39,934 NULL (  9.1%)

📊 RETURNS:
--------------------------------------------------------------------------------
   refunded_amount         4,008 NULL (  9.2%)

✅ SEKTION 5 ABGESCHLOSSEN: Initial Fill & Normalisierung

📝 Zusammenfassung:
   ✅ Text-Felder mit 'UNKNOWN' gefüllt (für NLP-Extraktion)
   ✅ Numerische Felder auf NULL normalisiert (NaN → NULL)
   ✅ Verbleibende NULLs sind GEWOLLT für spätere Berechnungen

➡️  Nächste Schritte:
   📌 Sektion 6: NLP-Extraktion für 'UNKNOWN' Werte
   📌 Sektion 7: NLP-Insertion für 'UNKNOWN' Werte



## 6. NLP: Extract Category, Brand from Description
Ziel: Fehlende Informationen wie Marke, Kategorie oder Preis automatisch aus der Produktbeschreibung extrahieren.

#### **Vorgehensweise**

Wir gehen in drei Hauptschritten vor, kombiniert aus  **Wordlist-Matching** und **Regex-Extraktion**:

#### 1. Bekannte Werte sammeln (Distinct)  
- Extrahiere alle vorhandenen **Marken** und **Kategorien** aus den Daten (`distinct()`).
- Erstelle daraus **Wordlists** (als *Sets*) → keine Duplikate, schnelle Vergleiche möglich.

#### 2. Neue Werte erkennen (Regex + Kontext)  
- Durchsuche die Produktbeschreibungen mit **Regex-Mustern**, um neue Marken, Kategorien oder Preisangaben zu identifizieren.  
- Ergänze die bestehenden Wordlists mit **neu entdeckten Begriffen**.

#### 3. Nicht-extrahierbare Werte kennzeichnen  
- Wenn kein Match gefunden wird, bleibt der Eintrag als **`'UNKNOWN'`** markiert.  


**Output**
- known_brands → vollständige, bereinigte Markenliste
- known_categories → vollständige, bereinigte Kategorienliste
- extracted_prices → Mapping von Produkt-ID zu ermitteltem Preis

### 6.1 Extrahiere bekannte Werte (Distinct)
Wir erstellen hier Sets für Marken, die wir bereits in den Spalten finden mit Distinct für Kategorien und Marken.
Ich habe hier **Sets gewählt anstelle Listen**, damit wir automatisch **keine Duplikate** haben und leichter die Sets miteinander mergen können mithilfe der Sets-Operations.

In [9]:
print("📊 Schritt 1: Extrahiere bekannte Werte aus vorhandenen Daten...\n")

# ============================================================================
# Bekannte Kategorien
# ============================================================================
# alle nicht-NULL, nicht-UNKNOWN Werte
known_categories_df = products_clean \
    .filter((col("category").isNotNull()) & (col("category") != "UNKNOWN")) \
    .select("category") \
    .distinct()

known_categories = set([row.category for row in known_categories_df.collect()])
print(f"✅ Bekannte Kategorien ({len(known_categories)}): {known_categories}")


# ============================================================================
# Bekannte Marken
# ============================================================================
# alle nicht-NULL, nicht-UNKNOWN Werte  
known_brands_df = products_clean \
    .filter((col("brand").isNotNull()) & (col("brand") != "UNKNOWN")) \
    .select("brand") \
    .distinct()

known_brands = set([row.brand for row in known_brands_df.collect()])
print(f"✅ Bekannte Marken ({len(known_brands)}): {known_brands}")


# ============================================================================
# Fallback falls Listen leer
# ============================================================================
# Falls die Listen leer sind (alle waren NULL), füge Fallback-Werte hinzu
if not known_brands:
    known_brands = {'Boss', 'Eastpak', 'Nike', 'Adidas', 'Puma', 'Reebok'}  
if not known_categories:
    known_categories = {'Sport', 'Schuhe', 'Bekleidung', 'Accessoires'} 


📊 Schritt 1: Extrahiere bekannte Werte aus vorhandenen Daten...

✅ Bekannte Kategorien (5): {'Schuhe', 'Accessoire', 'Sport', 'Kleidung', 'Zubehör'}
✅ Bekannte Marken (9): {'Puma', 'Levis', 'Boss', 'Esprit', 'Addidas', 'FILA', 'Eastpak', 'Bench', 'Nike'}


### 6.2 Erstelle Regex-Patterns
Hier habe ich Word-Patterns geschrieben wie zB: nimm alle Worte, die nach "von", "Hersteller" oder "Marke" um neue Marken oder Kategorien herauszufinden, die sich vielleicht in den Beschreibungen befinden.

In [10]:
print("=" * 80)
print("🔍 Schritt 2: Erstelle Regex-Patterns für Extraktion")
print("=" * 80)
print()

# ============================================================================
# Brand Patterns - Deutsche Sprache
# ============================================================================
brand_patterns = [
    r'[Vv]on\s+([A-Za-zäöüÄÖÜß]+)',           # "von Nike"
    r'[Mm]arke[:\s]+([A-Za-zäöüÄÖÜß]+)',      # "Marke: Adidas" / "Marke Puma"
    r'[Hh]ersteller[:\s]+([A-Za-zäöüÄÖÜß]+)', # "Hersteller: Boss"
    r'[Mm]arke\s+([A-Za-zäöüÄÖÜß]+)',         # "Marke Eastpak"
]

print(f"🏷️ Brand-Patterns: {len(brand_patterns)} Varianten")
for i, pattern in enumerate(brand_patterns, 1):
    print(f"   {i}. {pattern}")

# ============================================================================
# Category Patterns - Deutsche Sprache
# ============================================================================
category_patterns = [
    r'[Kk]ategorie[:\s]+([A-Za-zäöüÄÖÜß]+)',      # "Kategorie: Sport"
    r'aus\s+der\s+[Kk]ategorie\s+([A-Za-zäöüÄÖÜß]+)', # "aus der Kategorie Sport"
    r'[Kk]ategorie\s+([A-Za-zäöüÄÖÜß]+)',         # "Kategorie Sport"
]

print(f"\n📦 Category-Patterns: {len(category_patterns)} Varianten")
for i, pattern in enumerate(category_patterns, 1):
    print(f"   {i}. {pattern}")

print()

🔍 Schritt 2: Erstelle Regex-Patterns für Extraktion

🏷️ Brand-Patterns: 4 Varianten
   1. [Vv]on\s+([A-Za-zäöüÄÖÜß]+)
   2. [Mm]arke[:\s]+([A-Za-zäöüÄÖÜß]+)
   3. [Hh]ersteller[:\s]+([A-Za-zäöüÄÖÜß]+)
   4. [Mm]arke\s+([A-Za-zäöüÄÖÜß]+)

📦 Category-Patterns: 3 Varianten
   1. [Kk]ategorie[:\s]+([A-Za-zäöüÄÖÜß]+)
   2. aus\s+der\s+[Kk]ategorie\s+([A-Za-zäöüÄÖÜß]+)
   3. [Kk]ategorie\s+([A-Za-zäöüÄÖÜß]+)



### 6.3 Finde NEUE Werte mit Regex

In [11]:
print("🔍 Schritt 3: Suche neue Werte in descriptions")
print("=" * 80)

# Sammle alle descriptions mit UNKNOWN values
descriptions_to_analyze = products_clean.filter(
    (col("brand") == "UNKNOWN") | 
    (col("category") == "UNKNOWN")
).select("product_id", "description").collect()

print(f"📝 Analysiere {len(descriptions_to_analyze)} descriptions...")
print()

# Finde neue Brands
new_brands = set()
for row in descriptions_to_analyze:
    desc = row.description
    for pattern in brand_patterns:
        match = re.search(pattern, desc)
        if match:
            brand = match.group(1)
            if brand not in known_brands:
                new_brands.add(brand)

# Finde neue Categories
new_categories = set()
for row in descriptions_to_analyze:
    desc = row.description
    for pattern in category_patterns:
        match = re.search(pattern, desc)
        if match:
            category = match.group(1)
            if category not in known_categories:
                new_categories.add(category)

print(f"✅ Neue Brands gefunden:     {len(new_brands)}")
if new_brands:
    print(f"   → {list(new_brands)[:5]}...")  # Zeige erste 5

print(f"✅ Neue Categories gefunden: {len(new_categories)}")
if new_categories:
    print(f"   → {list(new_categories)[:5]}...")

print()

🔍 Schritt 3: Suche neue Werte in descriptions
📝 Analysiere 210 descriptions...

✅ Neue Brands gefunden:     4
   → ['klein', 'kommen', 'unbekannt', 'Helles']...
✅ Neue Categories gefunden: 1
   → ['Sonstiges']...



### 6.4 Merge: Bekannt + Neu = Finale Sets

In [12]:
print("🔗 Schritt 4: Erstelle finale Listen (bekannt + neu)")
print("=" * 80)

final_brands = known_brands | new_brands  # Set union
final_categories = known_categories | new_categories

print(f"📊 Finale Listen:")
print(f"   Brands:      {len(final_brands)} total")
print(f"                ({len(known_brands)} bekannt + {len(new_brands)} neu)")
print(f"   Categories:  {len(final_categories)} total")
print(f"                ({len(known_categories)} bekannt + {len(new_categories)} neu)")
print()


🔗 Schritt 4: Erstelle finale Listen (bekannt + neu)
📊 Finale Listen:
   Brands:      13 total
                (9 bekannt + 4 neu)
   Categories:  6 total
                (5 bekannt + 1 neu)



### 6.5 Extrahiere Preise

In [13]:
print("💶 Schritt 5: Extrahiere Preise aus descriptions")
print("=" * 80)

print("🔍 Strategie: Suche Zahl + 'Euro' oder '€'")
print("   ✅ Gültig:   '49,99 Euro', '34.50€', '199 EUR'")
print("   ❌ Ungültig: 'XX,XX Euro' (Platzhalter)")
print()

extracted_prices = {}  # product_id → price

# Hole alle Produkte ohne Preis
descriptions_with_null_price = products_clean.filter(
    col("price").isNull()
).select("product_id", "description").collect()

print(f"📝 Analysiere {len(descriptions_with_null_price)} descriptions...")
print()

# Simple Regex: Zahl (mit oder ohne Komma/Punkt) + Euro/€
# Beispiele: "49,99 Euro", "49.99€", "199 EUR", "34€"
price_pattern = r'(\d{1,4})[,.]?(\d{0,2})\s*(?:Euro|EUR|€)'

for row in descriptions_with_null_price:
    product_id = row.product_id
    desc = row.description
    
    # Suche nach Pattern
    match = re.search(price_pattern, desc, re.IGNORECASE)
    
    if match:
        main = match.group(1)      # z.B. "49"
        cents = match.group(2)     # z.B. "99" oder ""
        
        # Check: Ist es "XX" (Platzhalter)?
        if main.upper() == "XX":
            continue  # Skip Platzhalter!
        
        try:
            # Baue Preis zusammen
            if cents:
                # "49" + "99" → 49.99
                price = float(f"{main}.{cents}")
            else:
                # "49" → 49.0
                price = float(main)
            
            # Validation: Realistischer Preis?
            if 0 < price < 10000:
                extracted_prices[product_id] = price
            
        except ValueError:
            # Falls Konvertierung fehlschlägt (z.B. bei "XX")
            continue

print(f"✅ {len(extracted_prices)} gültige Preise gefunden")

# Zeige Beispiele
if extracted_prices:
    print("\n📊 Beispiele extrahierter Preise:")
    for pid, price in list(extracted_prices.items())[:5]:
        print(f"   Product {pid}: {price:.2f}€")
else:
    print("\n⚠️  Keine gültigen Preise gefunden (alle waren Platzhalter)")

print()

💶 Schritt 5: Extrahiere Preise aus descriptions
🔍 Strategie: Suche Zahl + 'Euro' oder '€'
   ✅ Gültig:   '49,99 Euro', '34.50€', '199 EUR'
   ❌ Ungültig: 'XX,XX Euro' (Platzhalter)

📝 Analysiere 46 descriptions...

✅ 0 gültige Preise gefunden

⚠️  Keine gültigen Preise gefunden (alle waren Platzhalter)



### 6.6 Zusammenfassung

In [14]:
print("=" * 80)
print("✅ SEKTION 7 ABGESCHLOSSEN: Extraction & Analysis")
print("=" * 80)
print()
print("📊 Ergebnisse:")
print(f"   final_brands:      {len(final_brands)} items")
print(f"   final_categories:  {len(final_categories)} items")
print(f"   extracted_prices:  {len(extracted_prices)} items")
print()
print("➡️  Nächster Schritt: Sektion 8 - Apply extracted values")
print()

✅ SEKTION 7 ABGESCHLOSSEN: Extraction & Analysis

📊 Ergebnisse:
   final_brands:      13 items
   final_categories:  6 items
   extracted_prices:  0 items

➡️  Nächster Schritt: Sektion 8 - Apply extracted values



## 7. Apply NLP Results (Update Phase)
Ich baue hier Bedingungen und Chains auf in Native Spark, weil dies schneller geht als UDFs. (Und weil ichs einfach mal lernen wollte)

**Ziel:**
- Match gegen Listen
- Update brand, category, product_name
- Fill prices
- Report statistics

### 7.1 Update products.BRAND

In [15]:
# ============================================================================
print("🏷️ Update BRAND...")
print("=" * 80)


brand_before = products_clean.filter(col("brand") == "UNKNOWN").count()
print(f"UNKNOWN brands vorher: {brand_before:,}")
print()

# Baue Condition für alle Brands: OR-Chain
brand_condition = None
for brand in final_brands:
    # Case-insensitive match
    current_condition = upper(col("description")).contains(brand.upper())
    
    if brand_condition is None:
        brand_condition = when(current_condition, lit(brand))
    else:
        brand_condition = brand_condition.when(current_condition, lit(brand))

# Fallback: UNKNOWN bleibt
brand_condition = brand_condition.otherwise(lit("UNKNOWN"))

# Update nur wo brand = UNKNOWN
products_clean = products_clean.withColumn(
    "brand",
    when(col("brand") == "UNKNOWN", brand_condition).otherwise(col("brand"))
)

brand_after = products_clean.filter(col("brand") == "UNKNOWN").count()
print(f"UNKNOWN brands nachher: {brand_after:,}")
print(f"✅ {brand_before - brand_after:,} brands gefüllt\n")


🏷️ Update BRAND...
UNKNOWN brands vorher: 166

UNKNOWN brands nachher: 53
✅ 113 brands gefüllt



### 7.2 Update products.CATEGORY

In [16]:
print("📦 Update CATEGORY...")
print("=" * 80)

category_before = products_clean.filter(col("category") == "UNKNOWN").count()
print(f"UNKNOWN categories vorher: {category_before:,}")
print()

# Baue Condition für alle Categories
category_condition = None
for category in final_categories:
    current_condition = upper(col("description")).contains(category.upper())
    
    if category_condition is None:
        category_condition = when(current_condition, lit(category))
    else:
        category_condition = category_condition.when(current_condition, lit(category))

category_condition = category_condition.otherwise(lit("UNKNOWN"))

products_clean = products_clean.withColumn(
    "category",
    when(col("category") == "UNKNOWN", category_condition).otherwise(col("category"))
)

category_after = products_clean.filter(col("category") == "UNKNOWN").count()
print(f"UNKNOWN categories nachher: {category_after:,}")
print(f"✅ {category_before - category_after:,} categories gefüllt\n")

📦 Update CATEGORY...
UNKNOWN categories vorher: 149

UNKNOWN categories nachher: 42
✅ 107 categories gefüllt



### 7.3 Update products.PRICES (aus extracted_prices dict)
Wissen, dass die meisten Preise aus den Beschreibungen zwar nur XX.XX€ sind, aber ich füge das hier trotzdem hinzu der Vollständigkeit halber.

In [17]:
if extracted_prices:
    print("💶 Update PRICES...")
    print("=" * 80)
    
    price_before = products_clean.filter(col("price").isNull()).count()
    print(f"NULL prices vorher: {price_before:,}")
    print()
    
    # Convert dict to DataFrame
    prices_df = spark.createDataFrame(
        [(k, v) for k, v in extracted_prices.items()],
        ["product_id", "extracted_price"]
    )
    
    # Join und update
    products_clean = products_clean.join(prices_df, "product_id", "left")
    
    products_clean = products_clean.withColumn(
        "price",
        when(col("price").isNull(), col("extracted_price")).otherwise(col("price"))
    )
    
    products_clean = products_clean.drop("extracted_price")
    
    price_after = products_clean.filter(col("price").isNull()).count()
    print(f"NULL prices nachher: {price_after:,}")
    print(f"✅ {price_before - price_after:,} prices gefüllt\n")
else:
    print("💶 Keine Preise aus description verfügbar")
    print()


💶 Keine Preise aus description verfügbar



## 8. Calculate Missing total_amount (SALES)

Fehlende `total_amount` Werte können berechnet werden: `price * quantity`
bei Sales fehlen die total_amount Werte, diese können mit Preis * Menge berechnet werden mit Join zu den products
join mit products um Preis zu bekommen.

Eigentlich ist diese Berechnung hier redundant, weil wir herausgefunden haben, dass wir keine Preise hinzufügen konnten, die gefehlt haben.
**Aber** da ich diese Berechnung schon bereits gemacht habe in der ersten Abgabe, füge ich sie hier als Übung nochmal hinzu :) 

In [18]:
print("=" * 80)
print("💰 SEKTION 8: BERECHNE FEHLENDE TOTAL_AMOUNT")
print("=" * 80)
print()

print("🎯 Strategie: total_amount = price × quantity")
print("   → Join mit products_clean um Preis zu bekommen")
print("   → Berechne nur wenn total_amount NULL ist")
print()

from pyspark.sql.functions import col, when

# ============================================================================
# 8.1 Count missing BEFORE
# ============================================================================
print("📊 Schritt 1: Zähle fehlende total_amount")
print("-" * 80)

missing_before = sales_clean.filter(col("total_amount").isNull()).count()
total_rows = sales_clean.count()

print(f"Total sales:           {total_rows:,}")
print(f"NULL total_amount:     {missing_before:,} ({missing_before/total_rows*100:.1f}%)")
print()

# ============================================================================
# 8.2 Join mit products um Preis zu bekommen
# ============================================================================
print("📊 Schritt 2: Join mit products_clean für Preis")
print("-" * 80)

# WICHTIG: Alias verwenden um Konflikte zu vermeiden!
sales_clean = sales_clean.join(
    products_clean.select(
        col("product_id"), 
        col("price").alias("product_price")  # ← Alias! Vermeidet Konflikt
    ),
    on="product_id",
    how="left"
)

# Prüfe: Wie viele haben KEINEN Preis?
no_price_count = sales_clean.filter(
    col("total_amount").isNull() & col("product_price").isNull()
).count()

print(f"Sales mit NULL total_amount: {missing_before:,}")
print(f"  Davon mit NULL price:      {no_price_count:,}")
print(f"  Davon berechenbar:         {missing_before - no_price_count:,}")
print()

if no_price_count > 0:
    print(f"⚠️  {no_price_count:,} sales können NICHT berechnet werden (kein Preis)")
    print("   → Diese bleiben NULL und werden in der nächsten Sektion behandelt")
    print()

# ============================================================================
# 8.3 Berechne total_amount = product_price × quantity
# ============================================================================
print("📊 Schritt 3: Berechne total_amount")
print("-" * 80)

# Berechne nur wenn:
# - total_amount ist NULL
# - UND product_price ist verfügbar (nicht NULL)
sales_clean = sales_clean.withColumn(
    "total_amount",
    when(
        col("total_amount").isNull() & col("product_price").isNotNull(),
        col("product_price") * col("quantity")  # Berechnung!
    ).otherwise(col("total_amount"))  # Behalte existierende Werte
)

# ============================================================================
# 8.4 Cleanup: Entferne temporäre Spalte
# ============================================================================
sales_clean = sales_clean.drop("product_price")

# ============================================================================
# 8.5 Count missing AFTER
# ============================================================================
print("\n📊 Schritt 4: Zähle fehlende total_amount NACH Berechnung")
print("-" * 80)

missing_after = sales_clean.filter(col("total_amount").isNull()).count()
calculated = missing_before - missing_after

print(f"Total sales:           {total_rows:,}")
print(f"NULL total_amount:     {missing_after:,} ({missing_after/total_rows*100:.1f}%)")
print(f"✅ {calculated:,} Werte erfolgreich berechnet!")
print()

# ============================================================================
# 8.6 Zeige Beispiele
# ============================================================================
print("📊 Beispiele BERECHNETER Werte:")
print("-" * 80)

sales_clean.filter(col("total_amount").isNotNull()) \
    .select("sale_id", "product_id", "quantity", "total_amount") \
    .show(10, truncate=False)

# Zeige auch nicht-berechenbare (falls vorhanden)
if missing_after > 0:
    print(f"\n⚠️  Beispiele NICHT-BERECHENBARER Werte:")
    print(f"    (Product hat keinen Preis - bleibt NULL)")
    print("-" * 80)
    
    # Rejoin kurz um Preis zu zeigen
    sales_with_price_info = sales_clean.join(
        products_clean.select("product_id", "price"),
        on="product_id",
        how="left"
    )
    
    sales_with_price_info.filter(col("total_amount").isNull()) \
        .select("sale_id", "product_id", "quantity", "price", "total_amount") \
        .show(5, truncate=False)

print()



💰 SEKTION 8: BERECHNE FEHLENDE TOTAL_AMOUNT

🎯 Strategie: total_amount = price × quantity
   → Join mit products_clean um Preis zu bekommen
   → Berechne nur wenn total_amount NULL ist

📊 Schritt 1: Zähle fehlende total_amount
--------------------------------------------------------------------------------
Total sales:           437,896
NULL total_amount:     39,934 (9.1%)

📊 Schritt 2: Join mit products_clean für Preis
--------------------------------------------------------------------------------
Sales mit NULL total_amount: 39,934
  Davon mit NULL price:      39,934
  Davon berechenbar:         0

⚠️  39,934 sales können NICHT berechnet werden (kein Preis)
   → Diese bleiben NULL und werden in der nächsten Sektion behandelt

📊 Schritt 3: Berechne total_amount
--------------------------------------------------------------------------------

📊 Schritt 4: Zähle fehlende total_amount NACH Berechnung
--------------------------------------------------------------------------------
Total 

### 8.7 Summary

In [19]:

print("=" * 80)
print("✅ SEKTION 8 ABGESCHLOSSEN")
print("=" * 80)
print()

print("📊 Zusammenfassung:")
print(f"   Berechnet:        {calculated:,} total_amount Werte")
print(f"   Verbleibend NULL: {missing_after:,}")
print()

if missing_after > 0:
    print("📝 Verbleibende NULLs:")
    print("   → Produkte ohne Preis")
    print("   → Werden in der nächsten Sektion behandelt")
    print()

print()

✅ SEKTION 8 ABGESCHLOSSEN

📊 Zusammenfassung:
   Berechnet:        0 total_amount Werte
   Verbleibend NULL: 39,934

📝 Verbleibende NULLs:
   → Produkte ohne Preis
   → Werden in der nächsten Sektion behandelt




### 8.8 Schlussfolgerung 
Wir sehen, dass die Null-Werte weiterhin bestehen bleiben. Das deutet darauf hin, dass die Preise selbst in der Beschreibung der Produktliste nicht vorhanden sind und wir somit kein total_amount berechnen können.
Hier müssten wir mit der Marketing / Buying Abteilung reden und fragen, ob wir von den verschiedenen Product_ids vollständige Tabellen haben um diese mit unsere product_clean.parquet zu mergen, oder ob es gewisse Stornos und Cancelations gab, sodass wir gewisse Daten rauslöschen könnten.

## 9. Null-Werte Quarantäne
Da wir noch nicht wissen, ob wir diese ungültigen Sales-Daten und Produktsdaten ohne Preise und Verkaufswert noch brauchen oder vielleicht später berichtigen könnten, haben wir uns dazu entschieden die  **NULL-Werte zu extrahieren**, in eignen Datenframe zu exportierten und diese **aus den existierenden Parquets zu entfernen**. Später können wir diese Quarantäne-Daten inkrementell wieder einfügen und verlieren somit auch keine wichtigen Daten. 
Dies ist auch fürs Logging und fürs Auditing vom Vorteil um nachher vielleicht sich nochmal die fehlerhaften Werte anzuschauen.

In [20]:
print("=" * 80)
print("🔒 NULL-WERTE QUARANTÄNE")
print("=" * 80)
print()

# Quarantäne-Pfad erstellen
QUARANTINE_PATH = "/app/data/data_quarantine/02_data_cleaning"
print(f"📂 Quarantäne-Pfad: {QUARANTINE_PATH}\n")

# ============================================================================
# 9.1 PRODUCTS - verbliebene Null-Werte von brand, category, price, product_name
# ============================================================================
print("🔍 PRODUCTS: Extrahiere Zeilen mit NULL-Werten...")

# Identifiziere Zeilen mit NULL in wichtigen Feldern
products_nulls = products_clean.filter(
    col("product_name").isNull() | 
    col("category").isNull() | 
    col("brand").isNull() | 
    col("price").isNull()
)

products_null_count = products_nulls.count()
print(f"   ⚠️  Gefunden: {products_null_count:,} Zeilen mit NULL-Werten")

if products_null_count > 0:
    # Speichere NULL-Werte
    products_nulls.toPandas().to_parquet(
        f"{QUARANTINE_PATH}/products_nulls.parquet", 
        index=False
    )
    print(f"   💾 Gespeichert: products_nulls.parquet")
    
    # Entferne NULL-Werte aus dem sauberen DataFrame
    products_clean = products_clean.filter(
        col("product_name").isNotNull() & 
        col("category").isNotNull() & 
        col("brand").isNotNull() & 
        col("price").isNotNull()
    )
    products_clean_count = products_clean.count()
    print(f"   ✅ Verbleibend: {products_clean_count:,} saubere Zeilen")
else:
    print("   ✅ Keine NULL-Werte gefunden!")

print()

# ============================================================================
# 9.2 SALES - verbliebende leere total_sales
# ============================================================================
print("🔍 SALES: Extrahiere Zeilen mit NULL-Werten...")

# Identifiziere Zeilen mit NULL in total_amount
sales_nulls = sales_clean.filter(col("total_amount").isNull())

sales_null_count = sales_nulls.count()
print(f"   ⚠️  Gefunden: {sales_null_count:,} Zeilen mit NULL total_amount")

if sales_null_count > 0:
    # Speichere NULL-Werte
    sales_nulls.toPandas().to_parquet(
        f"{QUARANTINE_PATH}/sales_nulls.parquet", 
        index=False
    )
    print(f"   💾 Gespeichert: sales_nulls.parquet")
    
    # Entferne NULL-Werte aus dem sauberen DataFrame
    sales_clean = sales_clean.filter(col("total_amount").isNotNull())
    sales_clean_count = sales_clean.count()
    print(f"   ✅ Verbleibend: {sales_clean_count:,} saubere Zeilen")
else:
    print("   ✅ Keine NULL-Werte gefunden!")

print()

# ============================================================================
# 9.3 RETURNS - missing refunded_amount
# ============================================================================
print("🔍 RETURNS: Extrahiere Zeilen mit NULL-Werten...")

# Identifiziere Zeilen mit NULL in refunded_amount
returns_nulls = returns_clean.filter(col("refunded_amount").isNull())

returns_null_count = returns_nulls.count()
print(f"   ⚠️  Gefunden: {returns_null_count:,} Zeilen mit NULL refunded_amount")

if returns_null_count > 0:
    # Speichere NULL-Werte
    returns_nulls.toPandas().to_parquet(
        f"{QUARANTINE_PATH}/returns_nulls.parquet", 
        index=False
    )
    print(f"   💾 Gespeichert: returns_nulls.parquet")
    
    # Entferne NULL-Werte aus dem sauberen DataFrame
    returns_clean = returns_clean.filter(col("refunded_amount").isNotNull())
    returns_clean_count = returns_clean.count()
    print(f"   ✅ Verbleibend: {returns_clean_count:,} saubere Zeilen")
else:
    print("   ✅ Keine NULL-Werte gefunden!")

print()

# ============================================================================
# 9.4 CUSTOMERS - Optional, weil wir wissen, dass der keine Null-Werte hat
# ============================================================================
print("🔍 CUSTOMERS: Extrahiere Zeilen mit NULL-Werten...")

# Identifiziere Zeilen mit NULL in wichtigen Feldern
customers_nulls = customers_clean.filter(
    col("customer_id").isNull() | 
    col("first_name").isNull() | 
    col("last_name").isNull() | 
    col("email").isNull() | 
    col("registration_date").isNull()
)

customers_null_count = customers_nulls.count()
print(f"   ⚠️  Gefunden: {customers_null_count:,} Zeilen mit NULL-Werten")

if customers_null_count > 0:
    # Speichere NULL-Werte
    customers_nulls.toPandas().to_parquet(
        f"{QUARANTINE_PATH}/customers_nulls.parquet", 
        index=False
    )
    print(f"   💾 Gespeichert: customers_nulls.parquet")
    
    # Entferne NULL-Werte aus dem sauberen DataFrame
    customers_clean = customers_clean.filter(
        col("customer_id").isNotNull() & 
        col("first_name").isNotNull() & 
        col("last_name").isNotNull() & 
        col("email").isNotNull() & 
        col("registration_date").isNotNull()
    )
    customers_clean_count = customers_clean.count()
    print(f"   ✅ Verbleibend: {customers_clean_count:,} saubere Zeilen")
else:
    print("   ✅ Keine NULL-Werte gefunden!")

print()


# ============================================================================
# 9.5 Quarantäne-Zusammenfassung
# ============================================================================
print("=" * 80)
print("📊 QUARANTÄNE-ZUSAMMENFASSUNG")
print("=" * 80)
print()

print("📦 In Quarantäne verschoben:")
print(f"   • PRODUCTS:  {products_null_count:,} Zeilen")
print(f"   • SALES:     {sales_null_count:,} Zeilen")
print(f"   • RETURNS:   {returns_null_count:,} Zeilen")
print(f"   • CUSTOMERS: {customers_null_count:,} Zeilen")
print()

total_quarantined = products_null_count + sales_null_count + returns_null_count + customers_null_count
print(f"📊 Total in Quarantäne: {total_quarantined:,} Zeilen")
print()

print("✅ Bereinigte DataFrames:")
print(f"   • products_clean:  {products_clean.count():,} Zeilen")
print(f"   • sales_clean:     {sales_clean.count():,} Zeilen")
print(f"   • returns_clean:   {returns_clean.count():,} Zeilen")
print(f"   • customers_clean: {customers_clean.count():,} Zeilen")
print()

print("💡 Hinweis:")
print("   Die quarantinierten Daten können später wieder eingefügt werden,")
print("   sobald die fehlenden Werte korrigiert wurden.")
print()

print("=" * 80)
print("✅ SEKTION 9 ABGESCHLOSSEN: NULL-Werte erfolgreich isoliert!")
print("=" * 80)
print()

🔒 NULL-WERTE QUARANTÄNE

📂 Quarantäne-Pfad: /app/data/data_quarantine/02_data_cleaning

🔍 PRODUCTS: Extrahiere Zeilen mit NULL-Werten...
   ⚠️  Gefunden: 46 Zeilen mit NULL-Werten
   💾 Gespeichert: products_nulls.parquet
   ✅ Verbleibend: 454 saubere Zeilen

🔍 SALES: Extrahiere Zeilen mit NULL-Werten...
   ⚠️  Gefunden: 39,934 Zeilen mit NULL total_amount
   💾 Gespeichert: sales_nulls.parquet
   ✅ Verbleibend: 397,962 saubere Zeilen

🔍 RETURNS: Extrahiere Zeilen mit NULL-Werten...
   ⚠️  Gefunden: 4,008 Zeilen mit NULL refunded_amount
   💾 Gespeichert: returns_nulls.parquet
   ✅ Verbleibend: 39,781 saubere Zeilen

🔍 CUSTOMERS: Extrahiere Zeilen mit NULL-Werten...
   ⚠️  Gefunden: 0 Zeilen mit NULL-Werten
   ✅ Keine NULL-Werte gefunden!

📊 QUARANTÄNE-ZUSAMMENFASSUNG

📦 In Quarantäne verschoben:
   • PRODUCTS:  46 Zeilen
   • SALES:     39,934 Zeilen
   • RETURNS:   4,008 Zeilen
   • CUSTOMERS: 0 Zeilen

📊 Total in Quarantäne: 43,988 Zeilen

✅ Bereinigte DataFrames:
   • products_clean: 

## 11. Save Cleaned Data

Speichere die bereinigten Daten als Parquet über pandas für die nächste Phase.

In [21]:
## 9. Save Cleaned Data

print("=" * 80)
print("💾 SPEICHERE BEREINIGTE DATEN")
print("=" * 80)
print()

OUTPUT_PATH = "/app/data/cleaned/2_data_cleaning"

print(f"Speichere nach: {OUTPUT_PATH}\n")

try:
    customers_clean.toPandas().to_parquet(f"{OUTPUT_PATH}/customers_clean.parquet")
    print("✅ customers_clean.parquet")
    
    products_clean.toPandas().to_parquet(f"{OUTPUT_PATH}/products_clean.parquet")
    print("✅ products_clean.parquet")
    
    sales_clean.toPandas().to_parquet(f"{OUTPUT_PATH}/sales_clean.parquet")
    print("✅ sales_clean.parquet")
    
    returns_clean.toPandas().to_parquet(f"{OUTPUT_PATH}/returns_clean.parquet")
    print("✅ returns_clean.parquet")
    
    print("\n🎉 Alle Daten gespeichert!")
    print(f"📂 Location: {OUTPUT_PATH}/")
    print("\n➡️  Bereit für Notebook 3: Data Quality Validation\n")
    
except Exception as e:
    print(f"❌ Fehler: {e}")

💾 SPEICHERE BEREINIGTE DATEN

Speichere nach: /app/data/cleaned/2_data_cleaning

✅ customers_clean.parquet
✅ products_clean.parquet
✅ sales_clean.parquet
✅ returns_clean.parquet

🎉 Alle Daten gespeichert!
📂 Location: /app/data/cleaned/2_data_cleaning/

➡️  Bereit für Notebook 3: Data Quality Validation



## 12. Cleaning Summary Report

In [22]:
print("\n" + "="*80)
print(" 📋 DATA CLEANING SUMMARY")
print("="*80 + "\n")

print("✅ **Completed Tasks:**")
print("   1. Removed duplicates (verification)")
print("   2. Filled missing values with intelligent defaults")
print("   3. Calculated missing total_amount in SALES")
print("   4. Extracted category from product descriptions")
print("   5. Extracted brand from product descriptions")
print("   6. Extracted product names from descriptions")
print("   7. Validated and corrected data types")
print("   8. Saved cleaned datasets")

print("\n📊 **Quality Improvements:**")
print("   - PRODUCTS: Reduced NULL values significantly")
print("   - SALES: Calculated all missing amounts")
print("   - All tables: Validated data types and ranges")

print("\n⏭️ **Next Steps:**")
print("   - Run 03_data_quality_validation.ipynb")
print("   - Implement Great Expectations validation")
print("   - Define data contracts")
print("   - Proceed to dimensional modeling")

print("\n" + "="*80)


 📋 DATA CLEANING SUMMARY

✅ **Completed Tasks:**
   1. Removed duplicates (verification)
   2. Filled missing values with intelligent defaults
   3. Calculated missing total_amount in SALES
   4. Extracted category from product descriptions
   5. Extracted brand from product descriptions
   6. Extracted product names from descriptions
   7. Validated and corrected data types
   8. Saved cleaned datasets

📊 **Quality Improvements:**
   - PRODUCTS: Reduced NULL values significantly
   - SALES: Calculated all missing amounts
   - All tables: Validated data types and ranges

⏭️ **Next Steps:**
   - Run 03_data_quality_validation.ipynb
   - Implement Great Expectations validation
   - Define data contracts
   - Proceed to dimensional modeling



In [23]:
# Spark Session beenden
spark.stop()
print("\n✅ Spark Session beendet. Data Cleaning abgeschlossen!")


✅ Spark Session beendet. Data Cleaning abgeschlossen!
